# Initialer Code

Folgender Code muss ausgeführt werden, um die Usecases zu validieren


In [1]:
endpoints = ["http://owid.de/api/fcs/"]; # Liste der Endpunkte, die abgefragt werden sollen.

from dataclasses import dataclass, field
from typing import List, Optional, Dict, Any
import xml.etree.ElementTree as ET
import requests

ns = {
    'sru': 'http://docs.oasis-open.org/ns/search-ws/sruResponse',
    'fcs': 'http://clarin.eu/fcs/resource',
    'hits': 'http://clarin.eu/fcs/dataview/hits',
    'lex': 'http://clarin.eu/fcs/dataview/lex'
}

@dataclass
class Hit:
    kind: Optional[str]
    value: str

@dataclass
class LexValue:
    text: str
    type: Optional[str] = None
    source: Optional[str] = None
    vocabValueRef: Optional[str] = None
    preferred: bool = False

@dataclass
class Record:
    pid: Optional[str]
    fragment_ref: Optional[str]
    hits: List[Hit] = field(default_factory=list)
    lex_fields: Dict[str, List[LexValue]] = field(default_factory=dict)
    record_position: Optional[int] = None

def _get_text(el):
    return el.text.strip() if el is not None and el.text is not None else None

def parse_sru(xml_text: str) -> List[Record]:
    root = ET.fromstring(xml_text)
    records: List[Record] = []
    for rec_el in root.findall('.//sru:record', ns):
        pos_el = rec_el.find('sru:recordPosition', ns)
        rec_pos = int(pos_el.text) if pos_el is not None and pos_el.text else None

        resource = rec_el.find('.//fcs:Resource', ns)
        pid = resource.get('pid') if resource is not None else None

        frag_el = rec_el.find('.//fcs:ResourceFragment', ns)
        frag_ref = frag_el.get('ref') if frag_el is not None else None

        hits_list: List[Hit] = []
        if frag_el is not None:
            for dv in frag_el.findall('fcs:DataView', ns):
                dv_type = dv.get('type','')
                if 'hits' in dv_type:
                    for hit_el in dv.findall('.//hits:Hit', ns):
                        hits_list.append(Hit(kind=hit_el.get('kind'), value=_get_text(hit_el) or ''))

        lex_fields: Dict[str, List[LexValue]] = {}
        if frag_el is not None:
            for dv in frag_el.findall('fcs:DataView', ns):
                dv_type = dv.get('type','')
                if 'lex' in dv_type:
                    entry = dv.find('lex:Entry', ns)
                    if entry is None:
                        continue
                    for field_el in entry.findall('lex:Field', ns):
                        ftype = field_el.get('type')
                        vals: List[LexValue] = []
                        for v in field_el.findall('lex:Value', ns):
                            vals.append(LexValue(
                                text=_get_text(v) or '',
                                type=v.get('type'),
                                source=v.get('source'),
                                vocabValueRef=v.get('vocabValueRef'),
                                preferred=(v.get('preferred') == 'true')
                            ))
                        if ftype:
                            lex_fields.setdefault(ftype, []).extend(vals)

        records.append(Record(pid=pid, fragment_ref=frag_ref, hits=hits_list, lex_fields=lex_fields, record_position=rec_pos))
    return records

def to_flat(record: Record) -> Dict[str, Any]:
    def first(ftype):
        vals = record.lex_fields.get(ftype)
        return vals[0].text if vals else None
    def all_texts(ftype):
        return [v.text for v in record.lex_fields.get(ftype, [])]

    landing = next((v.text for v in record.lex_fields.get('ref',[]) if v.type == 'landingpage'), None)
    return {
        'pid': record.pid,
        'ref': record.fragment_ref,
        'lemma': first('lemma'),
        'entryId': first('entryId'),
        'landingpage': landing,
        'pos': all_texts('pos'),
        'gender': all_texts('gender'),
        'segmentation': first('segmentation'),
        'definition': first('definition'),
        'citation_examples': [{'source': v.source, 'text': v.text} for v in record.lex_fields.get('citation', [])],
        'hits': [{'kind': h.kind, 'value': h.value} for h in record.hits],
        'recordPosition': record.record_position
    }

def fetch_and_parse(url: str, timeout: int = 10) -> List[Dict[str, Any]]:
    resp = requests.get(url, timeout=timeout, headers={'Accept': 'application/xml'})
    resp.raise_for_status()
    raw = parse_sru(resp.text)
    return [to_flat(r) for r in raw]

def _fetch_page_raw(url: str, timeout: int = 10):
    """Fetch a single SRU page and return (List[Record], nextRecordPosition, numberOfRecords)."""
    resp = requests.get(url, timeout=timeout, headers={'Accept': 'application/xml'})
    resp.raise_for_status()
    root = ET.fromstring(resp.text)
    records = parse_sru(resp.text)

    nr_el = root.find('.//sru:numberOfRecords', ns)
    next_el = root.find('.//sru:nextRecordPosition', ns)

    number_of_records = int(nr_el.text) if nr_el is not None and nr_el.text and nr_el.text.isdigit() else None
    next_record_pos = int(next_el.text) if next_el is not None and next_el.text and next_el.text.isdigit() else None

    return records, next_record_pos, number_of_records

def search_only_total_count(query: str) -> Optional[int]:
    """Return only the total number of results for `query` without fetching records."""
    for endpoint in endpoints:
        url = f"{endpoint}?queryType=lex&query={query}&startRecord=1&maximumRecords=1"
        _, _, number_of_records = _fetch_page_raw(url)
        if number_of_records is not None:
            return number_of_records
    return None

def search_first_page(query: str, maximum_records_per_request: int = 25) -> List[Dict[str, Any]]:
    """Return only the first page of results for `query`."""
    all_results: List[Dict[str, Any]] = []
    for endpoint in endpoints:
        url = f"{endpoint}?queryType=lex&query={query}&startRecord=1&maximumRecords={maximum_records_per_request}"
        page_records, _, _ = _fetch_page_raw(url)
        all_results.extend([to_flat(r) for r in page_records])
    return all_results

def search_all(query: str, maximum_records_per_request: int = 1000) -> List[Dict[str, Any]]:
    """Return all results for `query` by paging with startRecord / nextRecordPosition.

    - `maximum_records_per_request`: controls page size sent to the SRU endpoint; function will still
      request subsequent pages until all records are retrieved.
    """
    all_results: List[Dict[str, Any]] = []

    for endpoint in endpoints:
        start = 1
        total = None
        while True:
            url = f"{endpoint}?queryType=lex&query={query}&startRecord={start}&maximumRecords={maximum_records_per_request}"
            page_records, next_pos, number_of_records = _fetch_page_raw(url)

            all_results.extend([to_flat(r) for r in page_records])

            if total is None and number_of_records is not None:
                total = number_of_records

            if not next_pos:
                break
            if total is not None and len(all_results) >= total:
                break
            if next_pos <= start:
                break

            start = next_pos

    return all_results

In [2]:
data = search_only_total_count("Berg")
print(data)
data = search_first_page("Berg")
print(len(data))
data = search_all("Berg")
print(len(data))

411
25
411


In [3]:
print(data)

[{'pid': 'https://doi.org/10.14618/wb-elex', 'ref': 'https://www.owid.de/artikel/20691', 'lemma': 'Berg', 'entryId': '20691', 'landingpage': 'https://www.owid.de/artikel/20691', 'pos': ['nomen', 'NOUN'], 'gender': ['maskulinum', 'Masc'], 'segmentation': None, 'definition': 'Mit Berg bezeichnet man eine große, massive Erhebung aus Erde und Gestein im Gelände.', 'citation_examples': [{'source': 'St. Galler Tagblatt, 07.04.2010, S. 6, Leute.', 'text': 'Der nepalesische Rekord-Bergsteiger Apa Sherpa ist gestern zum 20. Mal zum Gipfel des Mount Everest aufgebrochen, wo er die Asche des ersten Bezwingers des Berges, des Neuseeländers Edmund Hillary, verstreuen will. Hillary hatte 1953 erstmals den höchsten Bergder Welt bezwungen.'}, {'source': 'Die Zeit, 24.11.1995, Wege für den Frieden, S. 79.', 'text': 'Der höchste Punkt des Kleinen Pal mißt 1866 Meter. Einen markanten Gipfel hat der Berg, ein karstiges zerklüftetes und zerfurchtes Hochplateau oberhalb der Baumgrenze, nicht. Durch jahrtaus